## Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio as ipy_audio
import librosa
import librosa.display

In [ ]:
import torch
from datasets import load_dataset
from datasets import Audio as hfd_audio
from transformers import pipeline
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor, AutoFeatureExtractor,
    AutoModelForAudioClassification, TrainingArguments, Trainer
)
import evaluate
# import gradio as gr

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## Prep

#### Dataset

In [ ]:
hub_dataset_id = "neerajaabhyankar/hindustani-raag-small"
dataset_id = hub_dataset_id  # Use hub ID for local version

In [ ]:
# hrs_full = load_dataset(dataset_id, revision="0dfb021e54e0e7489b90a47e23ef15f34fa740ec")
hrs_full = load_dataset(dataset_id)  # from drive path
hrs = hrs_full["train"].train_test_split(seed=42, shuffle=True, train_size=0.8, test_size=0.2, stratify_by_column="label") # train-val split
del hrs_full
dataset_name = dataset_id.split("/")[-1]

In [ ]:
# TEMP: v small dataset: Bageshri v/s Bheempalasi
SUBSET_IDS = [2, 8]
hrs_filtered = hrs.filter(lambda example: example["label"] in SUBSET_IDS)
hrs = hrs_filtered

#### ID2Label

In [ ]:
class_labels = hrs["train"].features["label"]

In [ ]:
id2label = {
    int(i): class_labels.int2str(i)
    for i in range(len(class_labels.names))
}
label2id = {v: k for k, v in id2label.items()}

In [ ]:
# # Temp: If using a smaller subset
SUBSET_LABELS = [id2label[s_id] for s_id in sorted(SUBSET_IDS)]

id2label = {i: label_name for i, label_name in enumerate(SUBSET_LABELS)}
label2id = {label_name: i for i, label_name in enumerate(SUBSET_LABELS)}

old_label_to_new_label = {old_label: new_label for new_label, old_label in enumerate(SUBSET_IDS)}

In [ ]:
def remap_dataset_labels(example):
    example["label"] = old_label_to_new_label[example["label"]]
    return example

hrs = hrs.map(remap_dataset_labels, num_proc=1)

#### Base Model

In [ ]:
model_id = "ntu-spml/distilhubert"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)

#### Prepare Dataset for Model

In [ ]:
sampling_rate = feature_extractor.sampling_rate  # 16000
max_duration = 90.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

In [ ]:
hrs = hrs.cast_column("audio", hfd_audio(sampling_rate=feature_extractor.sampling_rate))

In [ ]:
hrs_encoded = hrs.map(
    preprocess_function, remove_columns=["audio"], batched=True, num_proc=1
)

Sanity Check

In [ ]:
print(id2label[hrs_encoded["train"][0]["label"]])
sample = hrs_encoded["train"][0]["input_values"]
ipy_audio(data=sample, rate=feature_extractor.sampling_rate)

#### If That Fails

In [ ]:
# For some reason, colab tries to read audio from the path. Getting rid of the paths...

In [ ]:
# new_hrs = {"train": [], "test": []}

In [ ]:
# for sample in hrs["test"]:
#     new_sample = sample.copy()  # Create a shallow copy of the sample
#     if "audio" in new_sample and "path" in new_sample["audio"]:
#         del new_sample['audio']['path']  # Delete the 'path' key
#     new_hrs["test"].append(new_sample)

In [ ]:
# for sample in hrs["train"]:
#     new_sample = sample.copy()  # Create a shallow copy of the sample
#     if 'audio' in new_sample and 'path' in new_sample['audio']:
#         print(new_sample['audio']['path'])
#         del new_sample['audio']['path']  # Delete the 'path' key
#     new_hrs["train"].append(new_sample)

In [ ]:
# new_hrs["train"] = []
# len(new_hrs["train"]), len(new_hrs["test"])

In [ ]:
# from datasets import Dataset, NamedSplit
# from typing import OrderedDict
# del hrs
# new_hrs = OrderedDict(new_hrs)
# new_hrs_ds_train = Dataset.from_list(new_hrs["train"])
# new_hrs_ds_test = Dataset.from_list(new_hrs["test"])
# del new_hrs

In [ ]:
# hrs_encoded_train = new_hrs_ds_train.map(
#     preprocess_function, remove_columns=["audio"], batched=True, num_proc=1
# )
# del new_hrs_ds_train
# hrs_encoded_test = new_hrs_ds_test.map(
#     preprocess_function, remove_columns=["audio"], batched=True, num_proc=1
# )
# del new_hrs_ds_test
# hrs_encoded = {
#     "train": hrs_encoded_train,
#     "test": hrs_encoded_test
# }
# del hrs_encoded_train, hrs_encoded_test

#### Iterable

In [ ]:
hrs_encoded_train = hrs_encoded["train"].to_iterable_dataset().with_format("torch")
hrs_encoded_test = hrs_encoded["test"].to_iterable_dataset().with_format("torch")

#### Prepare Model for Finetuning

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    label2id=label2id,
    id2label=id2label,
    # torch_dtype=torch.bfloat16,
).to("mps")
model_name = model_id.split("/")[-1]

In [ ]:
model.device

In [ ]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
## Train settings ##

batch_size = 2
gradient_accumulation_steps = 1
num_train_epochs = 50
# max_steps = 1000  # if itrable dataset

training_args = TrainingArguments(
    f"{model_name}-finetuned-{dataset_name}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    # max_steps=max_steps,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    # fp16=True,
    push_to_hub=False,
)

## FineTune

In [ ]:
import os
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
trainer = Trainer(
    model,
    training_args,
    # train_dataset=hrs_encoded_train,
    # eval_dataset=hrs_encoded_test,
    train_dataset=hrs_encoded["train"],
    eval_dataset=hrs_encoded["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import pandas as pd
df = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(10, 6))
df[df['loss'].notna()].plot(x='epoch', y='loss', ax=ax, label='Train Loss', color='gold')
df[df['eval_loss'].notna()].plot(x='epoch', y='eval_loss', ax=ax, label='Eval Loss', color='tomato', marker='o')


## Plot

In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
preds = trainer.predict(hrs_encoded["test"])
pred_labels = np.argmax(preds.predictions, axis=1)
true_labels = hrs_encoded["test"]["label"]
# Fix: Use the remapped id2label for the confusion matrix
labels = [id2label[i] for i in sorted(id2label.keys())]

cm = confusion_matrix(true_labels, pred_labels, labels=list(range(len(labels))))

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.show()

## Save

In [ ]:
model.save_pretrained("./models/distilhubert-finetuned-bageshree-bheempalas-only-bs2")

Danger Zone

In [ ]:
# trainer.push_to_hub(**kwargs)

## Use [Recorded Sample] via Gradio

In [ ]:
# !apt-get update
# !apt-get install -y portaudio19-dev

In [ ]:
import soundfile as sf
import sounddevice as sd
import time
import io

In [ ]:
def record_audio(duration=5, sr=16000): # duration in seconds
    print(f"Recording for {duration} seconds...")
    # sd.query_devices() # uncomment to check available devices
    audio_data = sd.rec(int(duration * sr), samplerate=sr, channels=1, dtype='float32')
    sd.wait()  # Wait until recording is finished
    print("Recording finished.")
    return audio_data.flatten()

print("Defined record_audio function.")

In [ ]:
def classify_audio_gradio(sampling_rate, audio_data):
    # The audio_data from gr.Audio(type='numpy') is already a numpy array
    # The classifier pipeline takes a numpy array directly
    # The `sampling_rate` argument from Gradio is explicitly passed but not used by the classifier here,
    # as the feature_extractor configured with the classifier already has a fixed sampling rate (16000).
    # We ensure the input audio is at the expected sampling rate by configuring gr.Audio appropriately.

    # Perform classification
    prediction = classifier(audio_data)

    # # Gradio expects a dictionary for gr.Label output
    # # The prediction is a list of dictionaries, e.g., [{'score': 0.99, 'label': 'Bageshree'}]
    # # We'll take the top prediction.
    # top_prediction = prediction[0]
    # label = top_prediction['label']
    # score = top_prediction['score']
    # return {label: score}

    # return all the predictions and scores
    return prediction


print("Defined classify_audio_gradio function.")

In [ ]:
iface = gr.Interface(
    fn=classify_audio_gradio,
    inputs=gr.Audio(sources=['microphone'], type='numpy', streaming=True),
    outputs=gr.Label(),
    title="Live Raag Classifier",
    description="Record your voice and get a real-time raag prediction!"
)

iface.launch(debug=False, share=True)
print("Gradio interface launched.")

## Use [Recorded MP3s]

In [ ]:
model = AutoModelForAudioClassification.from_pretrained("./models/distilhubert-finetuned-bageshree-bheempalas-only")

In [ ]:
classifier = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor,
    label2id=label2id, id2label=id2label,
)

In [ ]:
for fn in os.listdir("adhoc-test-clips"):
    if not fn.endswith(".mp3"):
        continue
    path = os.path.join("adhoc-test-clips", fn)
    # load soundfile mp3
    audio_data, _ = librosa.load(path, sr=feature_extractor.sampling_rate)
    # pass through the classifier
    prediction = classifier(audio_data)
    print(f"filename = {fn}")
    print(prediction)
    print("\n")